# Загрузка и работа с макроданными

Этот ноутбук демонстрирует:
1. Загрузку макро-факторов из Excel шаблона
2. Визуализацию макро-факторов
3. Анализ эконометрических моделей
4. Экспорт макроданных в Excel
5. Примеры конфигурации

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Optional

# Добавляем корень проекта в путь
ROOT = Path().absolute().parent.parent.parent
sys.path.insert(0, str(ROOT))

from engine.database.data_mart import get_data_mart
from tools.load_macro_excel import main as load_macro_main
from tools.export_macro_factors_to_excel import export_macro_factors_to_excel

print(f"Корень проекта: {ROOT}")
print(f"Python путь: {sys.path[0]}")

## 1. Загрузка макроданных из Excel

In [ ]:
company = "us_steel"
excel_path = ROOT / "companies" / company / "data" / "macro" / f"{company}_macro.xlsx"

# Проверяем наличие файла
if excel_path.exists():
    print(f"✅ Файл найден: {excel_path}")
    
    # Загружаем макроданные
    # Используем argparse для вызова функции main
    import argparse
    
    # Создаем аргументы
    args = argparse.Namespace(
        root=ROOT,
        company=company,
        file=excel_path,
        config=ROOT / "companies" / company / "configs" / "excel_loader.yaml",
        scope="auto",
        source="manual",
        version=None,
        dry_run=False,
        verbose=True
    )
    
    # Загружаем (раскомментируйте для реальной загрузки)
    # load_macro_main()
    print("📊 Для загрузки раскомментируйте строку выше")
else:
    print(f"⚠️  Файл не найден: {excel_path}")
    print("Создайте файл или используйте export_macro_factors_to_excel для экспорта из БД")

## 2. Загрузка макроданных из БД и визуализация

In [ ]:
# Создаем DataMart
mart = get_data_mart(ROOT, company)

try:
    # Получаем список макро-факторов из конфигурации
    import yaml
    project_cfg = yaml.safe_load(
        (ROOT / "companies" / company / "configs" / "project.yaml").read_text(encoding="utf-8")
    )
    
    macro_factors = project_cfg.get("macro_forecast", {}).get("factors", [])
    print(f"Макро-факторы из конфигурации: {macro_factors}")
    
    # Загружаем данные для каждого фактора
    macro_data = {}
    for factor in macro_factors:
        try:
            data = mart.get_macro_factor(factor)
            if data:
                macro_data[factor] = data
                print(f"✅ {factor}: {len(data)} точек данных")
            else:
                print(f"⚠️  {factor}: данные не найдены")
        except Exception as e:
            print(f"❌ {factor}: ошибка загрузки - {e}")
    
    print(f"\nВсего загружено факторов: {len(macro_data)}")
    
finally:
    mart.close()

## 3. Визуализация макро-факторов

In [ ]:
# Настройка стиля
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Создаем DataFrame для визуализации
if macro_data:
    # Преобразуем в DataFrame
    dfs = []
    for factor, data in macro_data.items():
        df = pd.DataFrame({
            'year': list(data.keys()),
            'value': list(data.values()),
            'factor': factor
        })
        dfs.append(df)
    
    macro_df = pd.concat(dfs, ignore_index=True)
    
    # Визуализация временных рядов
    fig, axes = plt.subplots(len(macro_data), 1, figsize=(12, 4 * len(macro_data)))
    if len(macro_data) == 1:
        axes = [axes]
    
    for idx, (factor, data) in enumerate(macro_data.items()):
        years = sorted(data.keys())
        values = [data[y] for y in years]
        
        axes[idx].plot(years, values, marker='o', linewidth=2, markersize=6)
        axes[idx].set_title(f"Макро-фактор: {factor}", fontsize=14, fontweight='bold')
        axes[idx].set_xlabel("Год", fontsize=12)
        axes[idx].set_ylabel("Значение", fontsize=12)
        axes[idx].grid(True, alpha=0.3)
        axes[idx].set_xlim(min(years) - 1, max(years) + 1)
    
    plt.tight_layout()
    plt.show()
    
    # Корреляционная матрица (если есть общие годы)
    if len(macro_data) > 1:
        # Создаем pivot таблицу
        pivot_df = macro_df.pivot(index='year', columns='factor', values='value')
        
        # Вычисляем корреляции
        corr_matrix = pivot_df.corr()
        
        # Визуализация корреляционной матрицы
        plt.figure(figsize=(10, 8))
        sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                    square=True, linewidths=1, cbar_kws={"shrink": 0.8})
        plt.title("Корреляционная матрица макро-факторов", fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()
else:
    print("Нет данных для визуализации")

## 4. Анализ эконометрических моделей

In [ ]:
# Загружаем историю revenue для анализа
mart = get_data_mart(ROOT, company)

try:
    # Получаем историю revenue
    hist_is = mart.get_history_income_statement(canonical=False)
    
    if not hist_is.empty and 'revenue' in hist_is['metric'].str.lower().values:
        revenue_row = hist_is[hist_is['metric'].str.lower() == 'revenue'].iloc[0]
        
        # Извлекаем годы и значения revenue
        year_cols = [col for col in hist_is.columns if str(col).isdigit() and len(str(col)) == 4]
        revenue_data = {}
        for year_col in year_cols:
            try:
                year = int(year_col)
                value = revenue_row[year_col]
                if pd.notna(value):
                    revenue_data[year] = float(value)
            except (ValueError, TypeError):
                continue
        
        print(f"Revenue данные: {len(revenue_data)} точек")
        
        # Анализ связи revenue с макро-факторами
        if revenue_data and macro_data:
            # Находим общие годы
            common_years = set(revenue_data.keys())
            for factor_data in macro_data.values():
                common_years &= set(factor_data.keys())
            
            if len(common_years) >= 3:
                common_years = sorted(list(common_years))
                
                # Создаем DataFrame для регрессии
                reg_data = {'year': common_years, 'revenue': [revenue_data[y] for y in common_years]}
                
                for factor, factor_data in macro_data.items():
                    reg_data[factor] = [factor_data.get(y, np.nan) for y in common_years]
                
                reg_df = pd.DataFrame(reg_data)
                reg_df = reg_df.dropna()
                
                if len(reg_df) >= 3:
                    # Вычисляем корреляции
                    print("\nКорреляции revenue с макро-факторами:")
                    for factor in macro_data.keys():
                        if factor in reg_df.columns:
                            corr = reg_df['revenue'].corr(reg_df[factor])
                            print(f"  {factor}: {corr:.3f}")
                    
                    # Визуализация связи
                    if len(macro_data) > 0:
                        fig, axes = plt.subplots(len(macro_data), 1, figsize=(10, 4 * len(macro_data)))
                        if len(macro_data) == 1:
                            axes = [axes]
                        
                        for idx, (factor, _) in enumerate(macro_data.items()):
                            if factor in reg_df.columns:
                                axes[idx].scatter(reg_df[factor], reg_df['revenue'], alpha=0.6, s=50)
                                axes[idx].set_xlabel(factor, fontsize=12)
                                axes[idx].set_ylabel('Revenue', fontsize=12)
                                axes[idx].set_title(f"Revenue vs {factor}", fontsize=14, fontweight='bold')
                                axes[idx].grid(True, alpha=0.3)
                        
                        plt.tight_layout()
                        plt.show()
                else:
                    print("Недостаточно данных для анализа")
            else:
                print(f"Недостаточно общих годов: {len(common_years)}")
        else:
            print("Нет данных для анализа")
    else:
        print("Revenue не найден в истории")
        
finally:
    mart.close()

## 5. Экспорт макроданных в Excel

In [ ]:
# Экспорт макроданных из БД в Excel
output_path = ROOT / "companies" / company / "data" / "macro" / f"{company}_macro_exported.xlsx"

try:
    export_macro_factors_to_excel(
        company=company,
        output_path=output_path
    )
    print(f"\n✅ Макроданные экспортированы в: {output_path}")
except Exception as e:
    print(f"❌ Ошибка экспорта: {e}")

## 6. Примеры конфигурации

In [ ]:
# Показываем конфигурацию макро-факторов
project_cfg = yaml.safe_load(
    (ROOT / "companies" / company / "configs" / "project.yaml").read_text(encoding="utf-8")
)

macro_forecast_cfg = project_cfg.get("macro_forecast", {})

print("Конфигурация макро-факторов:")
print(f"  Профиль: {macro_forecast_cfg.get('profile', 'N/A')}")
print(f"  Конфиг: {macro_forecast_cfg.get('config', 'N/A')}")
print(f"  Факторы: {macro_forecast_cfg.get('factors', [])}")
print(f"  File map: {macro_forecast_cfg.get('file_map', {})}")
print(f"  Search paths: {macro_forecast_cfg.get('search_paths', [])}")

# Показываем конфигурацию revenue моделирования
revenue_cfg = project_cfg.get("model", {}).get("standard", {}).get("revenue", {})

print("\nКонфигурация revenue моделирования:")
print(f"  Драйверы: {revenue_cfg.get('drivers', [])}")
print(f"  Use Elastic Net: {revenue_cfg.get('use_elastic_net', False)}")
print(f"  L1 Ratio: {revenue_cfg.get('l1_ratio', 'N/A')}")
print(f"  Segment Modeling: {revenue_cfg.get('segment_modeling', False)}")
print(f"  Segment Volume Method: {revenue_cfg.get('segment_volume_method', 'N/A')}")
print(f"  Segment Price Method: {revenue_cfg.get('segment_price_method', 'N/A')}")
print(f"  Segment Volume Factors: {revenue_cfg.get('segment_volume_factors', [])}")
print(f"  Segment Price Factors: {revenue_cfg.get('segment_price_factors', [])}")